

# Public transport assignment with Optimal Strategies

In this example, we import a GTFS feed to our model, create a public transport network, create project match connectors, and perform a Spiess & Florian assignment. [Click here](https://doi.org/10.1016/0191-2615(89)90034-9) 
to check out the article.

We use data from Coquimbo, a city in La Serena Metropolitan Area in Chile.


.. admonition:: References

  * `transit_assignment`



.. seealso::
    Several functions, methods, classes and modules are used in this example:

    * :func:`aequilibrae.transit.Transit`
    * :func:`aequilibrae.transit.TransitGraphBuilder`
    * :func:`aequilibrae.paths.TransitClass`
    * :func:`aequilibrae.paths.TransitAssignment`
    * :func:`aequilibrae.matrix.AequilibraeMatrix`



In [1]:
# Imports for example construction
from uuid import uuid4
from os.path import join
from tempfile import gettempdir

from aequilibrae.transit import Transit
from aequilibrae.utils.create_example import create_example

In [2]:
# Let's create an empty project on an arbitrary folder.

##### CHANGE TEMP DIR #####
examples_dir = 'temp_examples'
#examples_dir = gettempdir()
fldr = join(examples_dir, uuid4().hex)
project = create_example(fldr, "coquimbo")

Let's create our ``Transit`` object.



In [3]:
data = Transit(project)

## Graph building
Let's build the transit network. We'll disable ``outer_stop_transfers`` and ``walking_edges`` 
because Coquimbo doesn't have any parent stations.

For the OD connections we'll use the ``overlapping_regions`` method and create some accurate line geometry later.
Creating the graph should only take a moment. By default zoning information is pulled from the project network. 
If you have your own zoning information add it using ``graph.add_zones(zones)`` then ``graph.create_graph()``. 



In [4]:
graph = data.create_graph(with_outer_stop_transfers=False, with_walking_edges=False, blocking_centroid_flows=False, connector_method="overlapping_regions")


# We drop geometry here for the sake of display.
graph.vertices.drop(columns="geometry")

,node_id,node_type,stop_id,line_id,line_seg_idx,taz_id
index,,,,,,
0,1,od,,,-1,1
1,2,od,,,-1,2
2,3,od,,,-1,3
3,4,od,,,-1,4
4,5,od,,,-1,5
...,...,...,...,...,...,...
362,363,alighting,10000000075,1_10001003000,31,
363,364,alighting,10000000076,1_10001003000,32,
364,365,alighting,10000000077,1_10001003000,33,


In [5]:
temp_df = graph.vertices
temp_df.reset_index(drop=True).drop(columns="geometry").node_type.unique()

['od', 'stop', 'boarding', 'alighting']
Categories (4, object): ['alighting', 'boarding', 'od', 'stop']

In [6]:
graph.edges

,link_id,link_type,line_id,stop_id,line_seg_idx,b_node,a_node,trav_time,freq,o_line_id,d_line_id,direction
index,,,,,,,,,,,,
0,1,on-board,1_10001001000,,0,212,290,86400.000000,inf,,,1
1,2,on-board,1_10001001000,,1,213,291,86400.000000,inf,,,1
2,3,on-board,1_10001001000,,2,214,292,86400.000000,inf,,,1
3,4,on-board,1_10001001000,,3,215,293,86400.000000,inf,,,1
4,5,on-board,1_10001001000,,4,216,294,86400.000000,inf,,,1
...,...,...,...,...,...,...,...,...,...,...,...,...
641,642,egress_connector,,,-1,166,112,630.543431,inf,,,1
642,643,egress_connector,,,-1,177,112,754.081137,inf,,,1
643,644,egress_connector,,,-1,178,112,386.262027,inf,,,1


In [7]:
graph.edges.link_type.unique()

['on-board', 'boarding', 'alighting', 'dwell', 'access_connector', 'egress_connector']
Categories (6, object): ['access_connector', 'alighting', 'boarding', 'dwell', 'egress_connector', 'on-board']

The graphs also also stored in the ``Transit.graphs`` dictionary. They are keyed by the 'period_id' they 
were created for. A graph for a different 'period_id' can be created by providing ``period_id=`` in the 
``Transit.create_graph`` call. You can view previously created periods with the ``Periods`` object.



In [8]:
periods = project.network.periods
periods.data

,period_id,period_start,period_end,period_description
0,1,0,86400,"Default time period, whole day"


## Connector project matching



In [9]:
project.network.build_graphs()

Now we'll create the line strings for the access connectors, this step is optinal but provides more accurate distance 
estimations and better looking geometry.

Because Coquimbo doesn't have many walking edges we'll match onto the ``"c"`` graph.



In [10]:
project.network.graphs

{'b': <aequilibrae.paths.graph.Graph at 0x1d3ce1d0b20>,
 'c': <aequilibrae.paths.graph.Graph at 0x1d3ce86b460>,
 't': <aequilibrae.paths.graph.Graph at 0x1d3ce842bb0>,
 'w': <aequilibrae.paths.graph.Graph at 0x1d3ce7b4bb0>}

In [11]:
# c for All motorized vehicles
graph.create_line_geometry(method="connector project match", graph="c")

c:\Users\germa\Documents\UQ\outer_loop\aequilibrae\aequilibrae\transit\transit_graph_builder.py:1215: UserWarning: In its current implementation, the "connector project match" method may take a while for large networks.
  warnings.warn(


## Saving and reloading
Lets save all graphs to the 'public_transport.sqlite' database.



In [12]:
data.save_graphs()

c:\Users\germa\Documents\UQ\outer_loop\aequilibrae\aequilibrae\transit\transit.py:99: UserWarning: Currently only a single transit graph can be saved and reloaded. Multiple graph support is plan for a future release.
  warnings.warn(


We can reload the saved graphs with ``data.load``. 
This will create new ``TransitGraphBuilder``\'s based on the 'period_id' of the saved graphs.
The graph configuration is stored in the 'transit_graph_config' table in 'project_database.sqlite' 
as serialised JSON.



In [13]:
data.load()

c:\Users\germa\Documents\UQ\outer_loop\aequilibrae\aequilibrae\transit\transit.py:113: UserWarning: Currently only a single transit graph can be saved and reloaded. Multiple graph support is plan for a future release. `period_ids` argument is currently ignored.
  warnings.warn(


Links and nodes are stored in a similar manner to the 'project_database.sqlite' database.

## Reading back into AequilibraE
You can create back in a particular graph via it's 'period_id'.



In [14]:
from aequilibrae.project.database_connection import database_connection
from aequilibrae.transit.transit_graph_builder import TransitGraphBuilder

In [15]:
pt_con = database_connection("transit")

graph_db = TransitGraphBuilder.from_db(pt_con, periods.default_period.period_id)
graph_db.vertices.drop(columns="geometry")

,node_id,node_type,stop_id,line_id,line_seg_idx,taz_id
0,1,od,,,-1,1
1,2,od,,,-1,2
2,3,od,,,-1,3
3,4,od,,,-1,4
4,5,od,,,-1,5
...,...,...,...,...,...,...
362,363,alighting,10000000075,1_10001003000,31,
363,364,alighting,10000000076,1_10001003000,32,
364,365,alighting,10000000077,1_10001003000,33,
365,366,alighting,10000000078,1_10001003000,34,


In [16]:
graph_db.edges

,link_id,link_type,line_id,stop_id,line_seg_idx,b_node,a_node,trav_time,freq,o_line_id,d_line_id,direction
0,1,on-board,1_10001001000,,0,212,290,86400.000000,inf,,,1
1,2,on-board,1_10001001000,,1,213,291,86400.000000,inf,,,1
2,3,on-board,1_10001001000,,2,214,292,86400.000000,inf,,,1
3,4,on-board,1_10001001000,,3,215,293,86400.000000,inf,,,1
4,5,on-board,1_10001001000,,4,216,294,86400.000000,inf,,,1
...,...,...,...,...,...,...,...,...,...,...,...,...
641,642,egress_connector,,,-1,166,112,1039.945114,inf,,,1
642,643,egress_connector,,,-1,177,112,1149.832226,inf,,,1
643,644,egress_connector,,,-1,178,112,508.359402,inf,,,1
644,645,egress_connector,,,-1,179,112,968.705083,inf,,,1


## Converting to a AequilibraE graph object
To perform an assignment we need to convert the graph builder into a graph.



In [17]:
#graph.edges['freq'] = 1

In [18]:
transit_graph = graph.to_transit_graph()
# this method should 3 skim cols (boardings, in vehicle-time, wait time)
# travel time (* special one)

# skim travel time for coquimbo network using this same notebook

## Mock demand matrix
We'll create a mock demand matrix with demand 1 for every zone.
We'll also need to convert from ``zone_id``\'s to ``node_id``\'s.



In [19]:
import numpy as np
from aequilibrae.matrix import AequilibraeMatrix

In [20]:
transit_graph.centroids

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133], dtype=uint32)

In [21]:
zones_in_the_model = len(transit_graph.centroids)

names_list = ['pt']

mat = AequilibraeMatrix()
mat.create_empty(zones=zones_in_the_model,
                 matrix_names=names_list,
                 memory_only=True)
mat.index = transit_graph.centroids[:]
mat.matrices[:, :, 0] = np.full((zones_in_the_model, zones_in_the_model), 1.0)
mat.computational_view()

In [22]:
mat.get_matrix('pt').shape

(133, 133)

In [23]:
mat.get_matrix('pt')

array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]])

## Hyperpath generation/assignment
We'll create a ``TransitAssignment`` object as well as a ``TransitClass``



In [24]:
from aequilibrae.paths import TransitAssignment, TransitClass

In [25]:
# Create the assignment class
assigclass = TransitClass(name="pt", graph=transit_graph, matrix=mat)
# use 3-d arr
# skim_matrices = {'trav_time': np.zeros_like(mat)}
assig = TransitAssignment()

assig.add_class(assigclass)

# We need to tell AequilbraE where to find the appropriate fields we want to use,  
# as well as the assignment algorithm to use.
assig.set_time_field("trav_time")
assig.set_frequency_field("freq")

assig.set_skimming_fields(["trav_time"])

assig.set_algorithm("os")

# When there's multiple matrix cores we'll also need to set the core to use for the demand.
assigclass.set_demand_matrix_core("pt")

# Let's perform the assignment for the transit classes added
assig.execute()


View the results



In [26]:
assig.classes[0].graph
for cls in assig.classes:
    print(cls._id)
    print(cls.graph)
    print(cls.matrix)

pt


In [27]:
df_graph = cls.graph.graph
df_graph

,link_id,a_node,b_node,direction,id,link_type,line_id,stop_id,line_seg_idx,trav_time,freq,o_line_id,d_line_id,geometry,__supernet_id__,__compressed_id__
0,479,18,158,1,0,egress_connector,,,-1,2146.077896,inf,,,b'\x01\x02\x00\x00\x00S\x00\x00\x00[S\xea\n\x8...,478,0
1,480,18,159,1,1,egress_connector,,,-1,632.392145,inf,,,"b""\x01\x02\x00\x00\x00!\x00\x00\x00[S\xea\n\x8...",479,1
2,481,18,183,1,2,egress_connector,,,-1,269.031753,inf,,,"b""\x01\x02\x00\x00\x00\x0e\x00\x00\x00[S\xea\n...",480,2
3,482,19,158,1,3,egress_connector,,,-1,1240.173389,inf,,,"b""\x01\x02\x00\x00\x00$\x00\x00\x00\x82\xa7\xe...",481,3
4,483,19,183,1,4,egress_connector,,,-1,2174.016494,inf,,,b'\x01\x02\x00\x00\x00V\x00\x00\x00\x82\xa7\xe...,482,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
641,74,362,284,1,641,on-board,1_10001003000,,31,86400.000000,inf,,,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00:H\xef\x...,73,638
642,75,363,285,1,642,on-board,1_10001003000,,32,86400.000000,inf,,,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00g\x94V\x...,74,639
643,76,364,286,1,643,on-board,1_10001003000,,33,86400.000000,inf,,,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00Q\xedf\x...,75,640
644,77,365,287,1,644,on-board,1_10001003000,,34,86400.000000,inf,,,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\xa8\xe1...,76,641


In [28]:
df_graph.freq.value_counts()

freq
inf         568
0.003333     78
Name: count, dtype: int64

In [29]:
check_df = df_graph[df_graph.freq != df_graph.freq[0]]
print(check_df.a_node.unique())
print(check_df.b_node.unique())

[211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228
 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246
 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264
 265 266 267 268 269 270 271 272 273 274 275 276 277 278 279 280 281 282
 283 284 285 286 287 288]
[133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150
 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168
 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186
 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204
 205 206 207 208 209 210]


In [30]:
check = assig.get_skim_results()
df = check[0]

# Check for values that are not 1.797693e+308 or 0
mask = (df != df.iloc[0,1]) & (df != 0)

# Filter the DataFrame
filtered_df = df[mask].dropna()
filtered_df

,1,2,3,4,5,6,7,8,9,10,...,124,125,126,127,128,129,130,131,132,133


In [31]:
check_vol = assig.results()
check_vol

,pt_volume
479,0.0
480,1.0
481,0.0
482,0.0
483,0.0
...,...
74,0.0
75,0.0
76,0.0
77,0.0


In [32]:
check_vol.pt_volume.unique()

array([ 0.,  1., 18., 24., 16., 15., 20., 19., 26., 22.,  9.,  3.,  7.,
        2., 28.,  4.,  8., 11., 17., 12.])

## Saving results
We'll be saving the results to another sqlite db called 'results_database.sqlite'. 
The 'results' table with 'project_database.sqlite' contains some metadata about each table in 
'results_database.sqlite'.



In [33]:
assig.save_results(table_name='hyperpath example')

Wrapping up



In [34]:
project.close()